In [0]:
# Azure ADLS Gen2 Configuration
storage_account_name = "<STORAGE_ACCOUNT_NAME>"
container_name = "landing-zone-2"
client_id = "<CLIENT_ID>"
tenant_id = "<TENANT_ID>"
client_secret = "<CLIENT_SECRET>"

# Configure OAuth authentication
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

adls_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/test.csv"

In [0]:
# List files in landing zone
files = dbutils.fs.ls("abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/")

for file in files:
    print(f"File: {file.name}, Size: {file.size} bytes")

Tên file: buyers-raw-2/, Kích thước: 0 bytes
Tên file: countries-raw-2/, Kích thước: 0 bytes
Tên file: processed_user_data/, Kích thước: 0 bytes
Tên file: sellers-raw-2/, Kích thước: 0 bytes
Tên file: to_processed_user_data/, Kích thước: 0 bytes
Tên file: users-raw-2/, Kích thước: 0 bytes


In [0]:
%fs ls abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/

path,name,size,modificationTime
abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/buyers-raw-2/,buyers-raw-2/,0,1773306338000
abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/countries-raw-2/,countries-raw-2/,0,1773306359000
abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/processed_user_data/,processed_user_data/,0,1773479282000
abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/sellers-raw-2/,sellers-raw-2/,0,1773306345000
abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/to_processed_user_data/,to_processed_user_data/,0,1773479270000
abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/users-raw-2/,users-raw-2/,0,1773306352000


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark = SparkSession.builder.appName("EcomDataPipeline").getOrCreate()

In [0]:
spark

In [0]:
# Read users data from landing zone
path = "abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/to_processed_user_data"
userDF = spark.read.parquet(path, header=True, inferSchema=True)
display(userDF)

identifierHash,type,country,language,socialNbFollowers,socialNbFollows,socialProductsLiked,productsListed,productsSold,productsPassRate,productsWished,productsBought,gender,civilityGenderId,civilityTitle,hasAnyApp,hasAndroidApp,hasIosApp,hasProfilePicture,daysSinceLastLogin,seniority,seniorityAsMonths,seniorityAsYears,countryCode
-7279641312655250028,user,Etats-Unis,en,3,8,0,0,0,0.0,0,0,F,2,mrs,False,False,False,True,709,3205,106.83,8.9,us
-1456013578740053406,user,Allemagne,de,3,8,0,0,0,0.0,0,0,F,2,mrs,False,False,False,True,709,3205,106.83,8.9,de
9006282053848196165,user,Suède,en,3,8,0,0,0,0.0,0,0,M,1,mr,True,False,True,True,689,3205,106.83,8.9,se
-7154634866120535654,user,Turquie,en,3,8,0,0,0,0.0,0,0,F,2,mrs,False,False,False,True,709,3205,106.83,8.9,tr
2858299215060733023,user,France,en,3,8,0,0,0,0.0,0,0,M,1,mr,True,False,True,True,709,3205,106.83,8.9,fr
-8370972521561479983,user,Royaume-Uni,en,3,8,0,0,0,0.0,0,0,F,2,mrs,False,False,False,True,709,3205,106.83,8.9,gb
-7877915015908472168,user,Royaume-Uni,en,3,8,4,0,0,0.0,0,0,F,2,mrs,False,False,False,True,591,3205,106.83,8.9,gb
7455841332634807036,user,Italie,fr,3,8,0,0,0,0.0,0,0,F,2,mrs,True,True,False,True,709,3205,106.83,8.9,it
4607255007288453096,user,Italie,fr,3,8,0,0,0,0.0,0,0,F,2,mrs,True,True,False,True,701,3205,106.83,8.9,it
-7302797141205914253,user,France,en,3,8,0,0,0,0.0,0,0,M,1,mr,True,False,True,True,703,3205,106.83,8.9,fr


In [0]:
%fs ls abfss://bronze@ecomadlskyle.dfs.core.windows.net/

path,name,size,modificationTime
abfss://bronze@ecomadlskyle.dfs.core.windows.net/buyers/,buyers/,0,1773384889000
abfss://bronze@ecomadlskyle.dfs.core.windows.net/countries/,countries/,0,1773384895000
abfss://bronze@ecomadlskyle.dfs.core.windows.net/sellers/,sellers/,0,1773384881000
abfss://bronze@ecomadlskyle.dfs.core.windows.net/users/,users/,0,1773384877000


In [0]:
bronze_base_path = "abfss://bronze@ecomadlskyle.dfs.core.windows.net/"

In [0]:
# Write users to Bronze layer as Delta
userDF.write.format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_base_path}/users")

In [0]:
# Read buyers data from landing zone
path = "abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/buyers-raw-2"
buyersDF = spark.read.parquet(path, header=True, inferSchema=True)

In [0]:
# Write buyers to Bronze layer as Delta
buyersDF.write.format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_base_path}/buyers")

In [0]:
# Read sellers data from landing zone
path = "abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/sellers-raw-2"
sellersDF = spark.read.parquet(path, header=True, inferSchema=True)

In [0]:
# Write sellers to Bronze layer as Delta
sellersDF.write.format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_base_path}/sellers")

In [0]:
# Read countries data from landing zone
path = "abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/countries-raw-2"
countriesDF = spark.read.parquet(path, header=True, inferSchema=True)

In [0]:
# Write countries to Bronze layer as Delta
countriesDF.write.format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_base_path}/countries")